# ಪಾಠ 11 - ಏಜೆಂಟ್-ಟು-ಏಜೆಂಟ್ (A2A) ಪ್ರೋಟೋಕಾಲ್


## ಸೆಟಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## ವಿಶೇಷ ಪ್ರಯಾಣ ಏಜೆಂಟ್ಸ್ ನಿರ್ಮಾಣ


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ವರ್ಕ್‌ಫ್ಲೋ ಮೂಲಕ ಬಹು-ಏಜೆಂಟ್ ಸಹಯೋಗ

ನಾವು ಮೂರು ಏಜೆಂಟ್‌ಗಳನ್ನು ಅನುಕ್ರಮವಾದ ವರ್ಕ್‌ಫ್ಲೋಗೆ ಸಂಪರ್ಕಿಸುತ್ತೇವೆ, ಅದು A2A ಸಂದೇಶ ಪ್ರವahitಮಾನದಂತೆ ಇದೆ:

1. **CurrencyExchangeAgent** ಬಳಕೆದಾರರ ವಿನಂತಿಯನ್ನು ಸ್ವೀಕರಿಸಿ ಕರೆನ್ಸಿ ಮಾರ್ಗದರ್ಶನವನ್ನು ಉತ್ಪಾದಿಸುತ್ತದೆ.
2. **ActivityPlannerAgent** ಸಮೃದ್ಧಾದ ಸಂದರ್ಭವನ್ನು ಸ್ವೀಕರಿಸಿ ಚಟುವಟಿಕೆ ಶಿಫಾರಸುಗಳನ್ನು ಸೇರಿಸುತ್ತದೆ.
3. **TravelManagerAgent** ಎರಡೂ ಇನ್‌ಪುಟ್‌ಗಳನ್ನು ಸಂಶ್ಲೇಷಿಸಿ ಅಂತಿಮ ಪ್ರಯಾಣ ವಿವರವನ್ನು ಸೃಷ್ಟಿಸುತ್ತದೆ.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ಉತ್ಪಾದನೆಯಲ್ಲಿ A2A ಅನ್ನು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವುದು

ಉತ್ಪಾದನಾ ಪರಿಸರದಲ್ಲಿ A2A ಪ್ರೊಟೋಕಾಲ್ ಶಕ್ತಿಶಾಲಿ ಕ್ರಾಸ್-ಸೇವಾ ಪರಿಸ್ಥಿತಿಗಳನ್ನು ಮುಕ್ತಗೊಳಿಸುತ್ತದೆ:

| ಸಾಮರ್ಥ್ಯ | ವಿವರಣೆ |
|---|---|
| **ಕ್ರಾಸ್-ಫ್ರೇಮ್‌ವರ್ಕ್ ಇಂಟರ್‌ಅಪ್‌** | ಒಂದು ಫ್ರೇಮ್‌ವರ್ಕ್‌ನೊಂದಿಗೆ ನಿರ್ಮಿತ ಏಜೆಂಟ್ ವಿದ್ಯಮಾನಗಳು ಇನ್ನೊಂದು ಯಾವುದೇ A2A-ಅನುರೂಪ ಫ್ರೇಮ್‌ವರ್ಕ್‌ನೊಂದಿಗೆ ನಿರ್ಮಿತ ಏಜೆಂಟ್‌ಗೆ ಕೆಲಸಗಳನ್ನು ಹಂಚಿಕೊಳ್ಳಬಹುದು, ಖಚಿತವಾಗಿ ಕ್ರಾಸ್-ಸಂಸ್ಥೆಯ ಇಂಟರ್ಪರೇಬಿಲಿಟಿ ಅನುಮತಿಸುತ್ತದೆ. |
| **ಸೇವೆ ಗಡಿ ರೇಖೆಗಳು** | ಏಜೆಂಟ್‌ಗಳು ಪ್ರತ್ಯೇಕ ಮೈಕ್ರೋಸರ್ವಿಸ್ಗಳು, ಕ್ಲೌಡ್ ಪ್ರದೇಶಗಳು ಅಥವಾ ವಿಭಿನ್ನ ಸಂಸ್ಥೆಗಳಲ್ಲಿಯೂ ವಾಸಿಸಬಹುದು ಆದರೆ ಸಹಕರಿಸುವಲ್ಲಿ ಸಲೀಸಾಗಿರುತ್ತಾರೆ. |
| **ಡೈನಾಮಿಕ್ ಅನ್ವೇಷಣೆ** | ಒಂದು ಸಂಗತಿಗೋಚಿ ಆಯೋಜಕರು ನಿರ್ವಹಣಾ ಸಮಯದಲ್ಲಿ ಏಜೆಂಟ್ ಕಾರ್ಡ್ ರಿಜಿಸ್ಟ್ರಿಯನ್ನು ಪ್ರಶ್ನಿಸಿ ಒಂದು ಉಪ-ಕಾರ್ಯಕ್ಕೆ ಸೂಕ್ತವಾದ ವಿಶೇಷಜ್ಞರನ್ನು ಕಂಡುಹಿಡಿಯಬಹುದು. |
| **ಸ್ಟ್ರೀಮಿಂಗ್ & ಪುಷ್ ಸೂಚನೆಗಳು** | A2A ಸರ್ವರ್-ಸೆಂಟ್ ಈವೆಂಟ್ಸ್ (SSE) ಮೂಲಕ实时 ಪ್ರಗತಿ ನವೀಕರಣಗಳು ಮತ್ತು ದೀರ್ಘಕಾಲಿಕ ಕಾರ್ಯಗಳಿಗೆ ಪುಷ್ ಸೂಚನೆಗಳನ್ನು ಬೆಂಬಲಿಸುತ್ತದೆ. |

ನಾವು ಮೇಲಿನ ಕಾರ್ಯಪ್ರವಾಹವು ಈ ಮಾದರಿಯ ಸರಳ, ಪ್ರಕ್ರಿಯೆಯಲ್ಲಿ ತಯಾರಾದ ಆವೃತ್ತಿಯಾಗಿದೆ. ನಿಜವಾದ
ನಿಯೋಜನೆಯಲ್ಲಿ ಪ್ರತಿ ಏಜೆಂಟ್ HTTP ಎಂಡ್‌ಪಾಯಿಂಟ್ ಅನ್ನು ಪ್ರಕಟಿಸುವುದು, ಏಜೆಂಟ್ ಕಾರ್ಡ್ ಅನ್ನು ಪ್ರಕಟಿಸುವುದು ಮತ್ತು
A2A JSON-RPC ಪ್ರೊಟೋಕಾಲ್ ಮೂಲಕ ಸಂವಹನ ಮಾಡಲು ಇರುತ್ತದೆ.


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಕಲಿತುಕೊಂಡಿದ್ದೀರಿ:

1. **A2A ಪ್ರೋಟೋಕಾಲ್ ಎಂದರೇನು** — ಏಜೆಂಟ್-ತಂದ-ಏಜೆಂಟ್ ಕಂಡುಹಿಡಿಯುವುದು, ಸಂದೇಶದ ವಿನಿಮಯ ಮತ್ತು ಕಾರ್ಯ ನಿರ್ವಹಣೆಗೆ ಆಯ್ದುಕೊಂಡ ಮಾದರಿ.
   ಮತ್ತು ಕಾರ್ಯ ನಿರ್ವಹಣೆಗೆ ತೆರೆಯಲ್ಪಟ್ಟ ಮಾನದಂಡ.
2. **ವಿಶಿಷ್ಟ ಏಜೆಂಟುಗಳನ್ನು ಹೇಗೆ ರಚಿಸಲಾಯುತ್ತದೆ** — ಕರೆನ್ಸಿ ವಿನಿಮಯ ಏಜೆಂಟ್, ಚಟುವಟಿಕೆ ಯೋಜಕ ಏಜೆಂಟ್,
   ಮತ್ತು ಟ್ರ್ಯಾವಲ್ ಮ್ಯಾನೇಜರ್ ಒರ್ಕೆಸ್ಟ್‌ರೇಟರ್.
3. **ಏಜೆಂಟುಗಳನ್ನು ಕಾರ್ಯ ಪ್ರಕ್ರಿಯೆಗೆ ಹೇಗೆ ಜೋಡಿಸುವುದು** — ಏಜೆಂಟುಗಳ ನಡುವೆ ಕ್ರಮವಾಗಿ ಸಂದೇಶದ ವಿನಿಮಯವನ್ನು ಮಾದರಿಗೊಳಿಸಲು `WorkflowBuilder` ಬಳಸುತಿವೆ.
   ಸಂದೇಶದ ವಿನಿಮಯದ ಕ್ರಮವನು `WorkflowBuilder` ಬಳಸಿ ಮಾದರಿಗೊಳಿಸುವುದು.
4. **A2A ಉತ್ಪಾದನೆಯಲ್ಲಿ ಹೇಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ** — ಸ್ಪಂದನಾತ್ಮಕ ಕಂಡುಹಿಡಿತ ಹಾಗೂ ಸ್ಟ್ರೀಮಿಂಗ್ ನವೀಕರಣಗಳೊಂದಿಗೆ ಕ್ರಾಸ್-ಫ್ರೇಮ್‌ವರ್ಕ್, ಕ್ರಾಸ್-ಸೇವಾ ಸಹಕಾರವನ್ನು ಸಧಿಸುವುದು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
